# 04. Context Engineering & Memory Deepening -- Static/Dynamic Context, InMemoryStore, Skills Pattern

Deep learning of LangGraph's context system and long-term memory (Store). Covers everything from static/dynamic runtime contexts to long-term memory based on semantic search, and Progressive Disclosure (Skills) patterns.

## Learning Objectives

- Understand the two-dimensional (Mutability x Lifetime) matrix of context engineering.
- Implement a static runtime context with `context_schema` + `@dataclass`
- Manage dynamic runtime context with `state_schema` and `AgentState` customizations
- Utilizes `InMemoryStore`’s namespace, put, get, and search APIs
- Build long-term memory based on semantic search
- Design by distinguishing between 3 types of memory (Semantic, Episodic, Procedural)
- Implement Progressive Disclosure using Skills pattern
- Compare hot path vs background memory writing strategies

## 4.1 Environment Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-4.1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Environment ready.")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 4.2 Context Engineering Overview

Context engineering is the design of systems that provide AI with “the right information, in the right format, at the right time.” Beyond simple prompt engineering, it is an architectural approach to programmatically assembling contexts at runtime.

There are two main reasons why agents fail:
1. Lack of LLM skills
2. **Lack of context or inappropriate context** (more frequent cause)

Therefore, context engineering is a core role for AI engineers and a fundamental solution to agent reliability.

### Two-dimensional matrix: Mutability x Lifetime

|  | **Runtime** (single run) | **Cross-conversation** (between sessions) |
|---|---|---|
| **Static** (immutable) | User ID, DB connection, tool definition | config files, etc. |
| **Dynamic** (variable) | Conversation history, intermediate results | User Preferences, Learned Memory |

### 3 context types

| Type | Mutability | Lifetime | Example | LangGraph Implementation |
|---|---|---|---|---|
| Static Runtime | Static | Single run | User ID, DB conn | `context_schema` |
| Dynamic Runtime (State) | Dynamic | Single run | Messages, interim results | `state_schema` |
| Dynamic Cross-conv (Store) | Dynamic | Cross-conversation | Affinity, Memory | `InMemoryStore` |

### 3 controllable context categories

| Category | Control object | Characteristics |
|---|---|---|
| **Model Context** | Instructions, message history, tool, response format | Transient |
| **Tool Context** | tool Access, read/write state, runtime context | Persistent |
| **Life-cycle Context** | Transformation between stages, Summary, guardrails | Persistent |

LangChain implements context engineering with a **middleware** mechanism. Middleware such as `@dynamic_prompt` and `@wrap_model_call` can be used to update context or control between lifecycle stages.

## 4.3 Static runtime context -- `context_schema` + `@dataclass`

Injects **unchanging** information into `context_schema` while the agent is running. Define the schema as `@dataclass`, and access it as `ToolRuntime[Context]` from tool.

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent

@dataclass
class UserContext:
    user_id: str
    role: str
    department: str

In [ ]:
@tool
def get_permissions(runtime: ToolRuntime[UserContext]) -> str:
    """View permissions based on the current user's role."""
    ctx = runtime.context
    perms = {"admin": "read,write,delete", "editor": "read,write"}
    return f"사용자 {ctx.user_id} ({ctx.department}): {perms.get(ctx.role, 'read')}"

In [ ]:
agent = create_agent(
    model=model, tools=[get_permissions], context_schema=UserContext,
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What are my rights?"}]},
    context=UserContext(user_id="u42", role="admin", department="eng"),
    config=lf_config,
)
print(result["messages"][-1].content)

### Key points

- You can use a type-safe context by passing `@dataclass` to `context_schema`
- Automatically injected as `runtime: ToolRuntime[Context]` type hint in tool function
- **read-only** and unchangeable while running
- Suitable data: User ID, DB connection, API key, session metadata

## 4.4 Dynamic runtime context -- `state_schema`, `AgentState` custom

This state **changes** as the agent processes messages and calls tool. Add custom fields by inheriting `AgentState`.

In [ ]:
from langchain.agents import AgentState

class RAGState(AgentState):
    """State including dynamic search context."""
    retrieved_docs: list[str]
    query_count: int

print(f"상태 키: {list(RAGState.__annotations__.keys())}")

In [ ]:
agent_with_state = create_agent(
    model=model, tools=[get_permissions],
    state_schema=RAGState, context_schema=UserContext,
)
result = agent_with_state.invoke(
    {"messages": [{"role": "user", "content": "hello!"}],
     "retrieved_docs": [], "query_count": 0},
    context=UserContext(user_id="u1", role="editor", department="sales"),
    config=lf_config,
)
print(f"최종 상태 키: {list(result.keys())}")

### Static vs Dynamic Comparison

| Category | Static Runtime (`context_schema`) | Dynamic Runtime (`state_schema`) |
|---|---|---|
| Whether to change | immutable (read-only) | variable (node ​​updates) |
| Delivery method | `context=` parameters | invoke input dict |
| Approach | `runtime.context.field` | `state["field"]` |
| suitable data | Authentication Information, Settings | Conversation history, intermediate results |

## 4.5 Long-term memory -- InMemoryStore native API

Use `InMemoryStore` for cross-conversation context. Long-term memory is per-user or app-level data that persists across sessions and threads.

### Storage structure
The memory is stored as a **JSON document**, organized hierarchically into **namespace**:
- **namespace**: Folder role to classify memory (e.g. `(user_id, "preferences")`)
- **key**: Unique identifier for each memory (e.g. `"theme"`)
- Namespaces usually include user IDs or organization IDs to facilitate information management.

### Basic API
| API | Description |
|---|---|
| `store.put(namespace, key, value)` | Save memory (upsert) |
| `store.get(namespace, key)` | Memory query by specific key |
| `store.search(namespace)` | Search all within a namespace |
| `store.search(namespace, filter={...})` | Search by filter conditions |

In production environments, you should use **DB-based Store** (e.g. PostgreSQL) instead of `InMemoryStore`.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
user_id = "user_42"
store.put((user_id, "preferences"), "theme", {"value": "dark"})
store.put((user_id, "preferences"), "language", {"value": "ko"})

item = store.get((user_id, "preferences"), "theme")
print(f"테마: {item.value}")

In [ ]:
items = store.search((user_id, "preferences"))
for item in items:
    print(f"  [{item.key}] = {item.value}")

filtered = store.search(
    (user_id, "preferences"), filter={"value": "dark"}
)
print(f"필터 결과: {len(filtered)}건")

## 4.6 Long-Term Memory -- Semantic Search

Setting the embedding function makes `InMemoryStore` support **semantic search**. Perform semantic-based similarity search with the `query` parameter.

In [ ]:
semantic_store = InMemoryStore(
    index={"embed": embeddings, "dims": 1536}
)
ns = ("user_42", "memories")
semantic_store.put(ns, "mem1", {"content": "Prefer pytest over unittest"})
semantic_store.put(ns, "mem2", {"content": "Use type hints for all functions"})
semantic_store.put(ns, "mem3", {"content": "Favorite food is sushi"})
semantic_store.put(ns, "mem4", {"content": "Works on the ML Infrastructure team"})
print("Four memories have been saved along with embedding.")

In [ ]:
results = semantic_store.search(
    ("user_42", "memories"), query="testing preferences", limit=2
)
for r in results:
    print(f"  [{r.key}] {r.value['content']}")

In [ ]:
results2 = semantic_store.search(
    ("user_42", "memories"), query="machine learning work", limit=2
)
for r in results2:
    print(f"  [{r.key}] {r.value['content']}")

### Basic Store vs Semantic Store comparison

| Features | `InMemoryStore()` | `InMemoryStore(index={...})` |
|---|---|---|
| Accurate key lookup | `get(ns, key)` | `get(ns, key)` |
| Filter Search | `search(ns, filter={...})` | `search(ns, filter={...})` |
| Semantic Search | Not possible | `search(ns, query="...", limit=N)` |
| Production | Use DB backend instead of `InMemoryStore` | PostgreSQL-based Store recommended |

## 4.7 Reading/Writing Store from tool -- `ToolRuntime.store`

You can access the Store from within the agent's tool to read and write user information. If you connect a Store to `create_agent(store=...)`, it will be automatically injected into `runtime.store`.

### Reading Pattern
Search for user information saved through `runtime.store` in tool. Both context and Store can be accessed with `ToolRuntime[Context]` type hints.

### Writing Pattern
Receives user input with the tool parameter and saves the memory as `store.put()`. This allows agents to permanently store information learned during a conversation.

### Key points
- `runtime.store`: Access Store instance
- `runtime.context`: Access static runtime context
- By combining Store and Context, you can systematically manage **"whose (context) information (store)"**

In [ ]:
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    """Search the saved information of the current user."""
    store = runtime.store
    user_id = runtime.context.user_id
    info = store.get(("users",), user_id)
    return str(info.value) if info else "User information not found."

In [ ]:
@tool
def save_preference(key: str, value: str, runtime: ToolRuntime[UserContext]) -> str:
    """Stores user preferences."""
    store = runtime.store
    user_id = runtime.context.user_id
    store.put((user_id, "preferences"), key, {"value": value})
    return f"선호도 저장됨: {key}={value}"

In [ ]:
memory_agent = create_agent(
    model=model,
    tools=[get_user_info, save_preference],
    context_schema=UserContext,
    store=semantic_store,
)
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Please save the theme as dark"}]},
    context=UserContext(user_id="u42", role="admin", department="eng"),
    config=lf_config,
)
print(result["messages"][-1].content)

## 4.8 3 types of memory: Semantic, Episodic, Procedural

Long-term memory is categorized into three types inspired by cognitive science. **Storage structure** and **utilization** are different for each type.

| Type | Description | Example | structure |
|---|---|---|---|
| **Semantic** | factual knowledge about entities | User preferences, profile information | Profile or Collection |
| **Episodic** | Memories of past experiences and events | Few-shot example, past action log | Collection |
| **Procedural** | Rules/Guidelines on how to do it | Fix system prompt, guidelines | Profile (list of rules) |

### Semantic Memory -- Profile vs Collection

Semantic memory has two approaches depending on the storage strategy:

| Approach | structure | Suitable for | Example |
|---|---|---|---|
| **Profile** | Single JSON document, continuously updated | Few well-known properties | `{"name": "Alice", "language": "Python", "preferred_style": "concise"}` |
| **Collection** | Multiple narrow documents, high recall | Open-end or large-scale knowledge | `[{"topic": "testing", "content": "Prefers pytest"}, ...]` |

### Episodic Memory
Record how you have behaved in similar situations in the past. It is used as a few-shot example, allowing the agent to learn from past experiences.

### Procedural Memory
Stores the agent's behavioral rules. This has the effect of dynamically modifying system prompts, allowing the agent to follow user-specific instructions.

In [ ]:
mem_store = InMemoryStore(index={"embed": embeddings, "dims": 1536})
uid = "user_42"

# Semantic -- Profile (single JSON)
mem_store.put((uid, "profile"), "main", {
    "name": "Alice", "language": "Python",
    "preferred_style": "concise",
})
# Semantic -- Collection (multiple docs)
mem_store.put((uid, "facts"), "f1", {"content": "pytest preferred"})

In [ ]:
# Episodic -- past experiences (few-shot)
mem_store.put((uid, "episodes"), "ep1", {
    "content": "SQL Optimization -> Use EXPLAIN ANALYZE",
})

# Procedural -- rules/guidelines
mem_store.put((uid, "procedures"), "rules", {
    "content": "Always includes error handling. Use logging.",
})
print("All three memory types have been saved.")

In [ ]:
# Episodic search: find similar past experiences
episodes = mem_store.search(
    (uid, "episodes"), query="database query help", limit=1
)
for ep in episodes:
    print(f"관련 에피소드: {ep.value['content']}")

## 4.9 Progressive Disclosure -- Skills pattern

Putting all the context in the prompt increases token cost and reduces accuracy. The Skills pattern is a Progressive Disclosure method that loads relevant information only when needed.

### Structure of Skill
Skill is a unit of knowledge consisting of `{name, description, content}`:
- **name**: Skill identifier (e.g. `"customers_schema"`)
- **description**: Short description (included in system prompt)
- **content**: Details (load on demand with `load_skill` tool)

### Strategies by Size

| size | strategy | Example |
|---|---|---|
| **<1K tokens** | Included directly in the system prompt | table names, high-level relationships |
| **1-10K tokens** | Load on demand with `load_skill` tool | Table schema, query patterns, best practices |
| **>10K tokens** | Load on demand with pagination | Large reference data, historical query logs |

### Action flow
1. **Middleware** injects the names and descriptions of all skills into the system prompt
2. The agent analyzes the question and determines the skills needed
3. Call `load_skill` tool to load details
4. Perform actions based on loaded content

### Advantages

| Advantages | Description |
|---|---|
| **Token Efficiency** | Load only the information needed for the current query |
| **Scalability** | DB with hundreds of tables is also supported |
| **Accuracy** | Provide detailed schema at the point you need it |
| **Cost Savings** | Reduce input tokens per request |

In [ ]:
skills = [
    {"name": "db_overview",
     "description": "High-level Overview for all tables",
     "content": "Table: customers, orders, products"},
    {"name": "customers_schema",
     "description": "Full schema of the customers table",
     "content": "CREATE TABLE customers (id INT PK, name VARCHAR)"},
]
SKILL_MAP = {s["name"]: s for s in skills}
print(f"스킬 {len(skills)}개 정의됨.")

In [ ]:
from langchain_core.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """Loads detailed information about the database skill."""
    skill = SKILL_MAP.get(skill_name)
    if skill is None:
        return f"찾을 수 없음. 사용 가능: {', '.join(SKILL_MAP.keys())}"
    return f"## {skill['name']}\n\n{skill['content']}"

## 4.10 Hot Path vs Background Memory Write

Depending on **when** you use memory, it will affect user response latency.

| method | Timing | Available immediately? | Delay Impact |
|---|---|---|---|
| **Hot path** | Real-time within conversation loop | Instantly (available next turn) | Increased response delay |
| **Background** | Separate asynchronous task | Delayed (Eventual Consistency) | No delay impact |

### Write Hot Path
Save memory inline within the agent loop. This is perfect when you need to use that memory in the very next turn. Example: When you need to immediately reflect the preferences that the user just shared.

### Background writing
Save memory as a separate process or asynchronous task. Used when Eventual Consistency is allowed and does not affect response delay. Examples: conversation pattern analysis, long-term learning data accumulation.

### Selection criteria
- Is an immediate recall necessary? -> **Hot path**
- Is reducing delay a priority? -> **Background**
- In most cases, background writing is preferred.

In [ ]:
from langgraph.store.base import BaseStore

# Hot path: write inline (adds latency)
def reflect_node(state, store: BaseStore):
    """Extract and store memory inline."""
    last_msg = state["messages"][-1].content
    store.put(("user", "reflections"), "latest", {"content": last_msg})
    return state

print("Hot path: Immediately save, available next turn.")

In [ ]:
import asyncio

# Background: write in separate async task
async def background_memory_writer(state, store: BaseStore):
    """Saves memory in the background (no delays)."""
    last_msg = state["messages"][-1].content
    await store.aput(
        ("user", "reflections"), "latest", {"content": last_msg}
    )
print("Background: Eventual consistency, no delay.")

## Summary

### Context Engineering Triad

| element | implementation | API |
|---|---|---|
| static runtime | `context_schema` + `@dataclass` | `runtime.context.field` |
| dynamic runtime | `state_schema` + `AgentState` | `state["field"]` |
| long term memory | `InMemoryStore` + `store=` | `store.put/get/search` |

### Memory 3 types

| Type | Use | Namespace example |
|---|---|---|
| Semantic | User Profile/Facts | `(user_id, "profile")`, `(user_id, "facts")` |
| Episodic | Past experience (few-shot) | `(user_id, "episodes")` |
| Procedural | Edit Rule/Prompt | `(user_id, "procedures")` |

### Best Practices

| Principle | Description |
|---|---|
| minimize static context | Include only what is needed for the current task |
| Namespace structuring | Use hierarchical namespace to avoid collisions |
| Semantic search priority | Embedding-based search is more scalable than exact matching |
| Background writing preference | Reduce delay to background when immediate recall is not required |
| Skills pattern application | Large-scale context can be found in Progressive Disclosure |

### Next Steps
→ **[05_agentic_rag.ipynb](./05_agentic_rag.ipynb)**: Learn Agentic RAG.